In [62]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import py360convert
import json
from skimage.feature import peak_local_max
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# 1. Configuración del Modelo
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512").to(device)
model.eval()

# Clases: 2:cielo, [4,12,17]:vegetación/cultivo, [6,9,13,29,94]:suelo, 15:persona
CLASSES = {"vegetacion": [4, 12, 17, 72], "cielo": [2], "suelo": [6, 9, 13, 29, 94], "persona": [15]}

def process_precise_y_ranges(img_path, output_filename="puntos_parametrizados.json", face_w=512):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: raise FileNotFoundError(f"Error: {img_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # Analizar Cubemap
    cube_dice = py360convert.e2c(img_rgb, face_w=face_w, cube_format='dice')
    faces_coords = [(face_w, 2*face_w, 2*face_w, 3*face_w), (face_w, 2*face_w, 0, face_w), (0, face_w, face_w, 2*face_w), (2*face_w, 3*face_w, face_w, 2*face_w), (face_w, 2*face_w, face_w, 2*face_w), (face_w, 2*face_w, 3*face_w, 4*face_w)]
    
    cube_maps = np.zeros((4, 3*face_w, 4*face_w), dtype=np.float32)

    for (y0, y1, x0, x1) in faces_coords:
        face = cube_dice[y0:y1, x0:x1, :]
        inputs = processor(images=face, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inputs).logits
            probs = F.softmax(F.interpolate(out, size=(face_w, face_w), mode='bilinear'), dim=1).squeeze().cpu().numpy()
        
        cube_maps[0, y0:y1, x0:x1] = np.sum(probs[CLASSES["vegetacion"]], axis=0)
        cube_maps[1, y0:y1, x0:x1] = np.sum(probs[CLASSES["cielo"]], axis=0)
        cube_maps[2, y0:y1, x0:x1] = np.sum(probs[CLASSES["suelo"]], axis=0)
        cube_maps[3, y0:y1, x0:x1] = np.sum(probs[CLASSES["persona"]], axis=0)

    final_maps = {}
    for i, cat in enumerate(["vegetacion", "cielo", "suelo", "persona"]):
        temp = np.repeat(cube_maps[i][:, :, np.newaxis], 3, axis=2)
        final_maps[cat] = py360convert.c2e(temp, h, w, cube_format='dice')[:, :, 0]

    # --- PARAMETRIZACIÓN ESTRICTA DE RANGOS Y ---
    
    # Bloqueo de personas previo
    person_mask = final_maps["persona"] > 0.3
    final_maps["vegetacion"][person_mask] = 0
    final_maps["suelo"][person_mask] = 0

    # Inicializar mapas limpios con las restricciones de Y solicitadas
    
    # CIELO: y <= 500
    sky_final = np.zeros_like(final_maps["cielo"])
    limit_sky = min(500, h)
    sky_final[:limit_sky, :] = final_maps["cielo"][:limit_sky, :]
    
    # CULTIVO (Vegetación): 1000 <= y <= 1300
    veg_final = np.zeros_like(final_maps["vegetacion"])
    if h > 1000:
        limit_veg_up = 1000
        limit_veg_down = min(1300, h)
        veg_final[limit_veg_up:limit_veg_down, :] = final_maps["vegetacion"][limit_veg_up:limit_veg_down, :]

    # SUELO: y >= 1900
    suelo_final = np.zeros_like(final_maps["suelo"])
    if h > 1900:
        suelo_final[1900:, :] = final_maps["suelo"][1900:, :]

    # --- EXTRACCIÓN DE PUNTOS ---
    results = {"puntos": {}}
    maps_to_extract = {
        "cielo": sky_final,
        "cultivo": veg_final,
        "suelo": suelo_final
    }

    for cat, m in maps_to_extract.items():
        # Usamos un umbral muy bajo (0.01) para encontrar puntos incluso en franjas estrechas
        coords = peak_local_max(m, min_distance=w//20, threshold_abs=0.01, num_peaks=5)
        
        results[cat] = [{"x": int(c[1]), "y": int(c[0]), "conf": round(float(m[c[0], c[1]]), 4)} for c in coords]

    with open(output_filename, 'w') as f:
        json.dump(results, f, indent=4)
    
    return output_filename

if __name__ == "__main__":
    process_precise_y_ranges("prueba2.0.jpg")
    print("Script finalizado. Rangos: Cielo (<500), Cultivo (1000-1300), Suelo (>1900).")

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 332.79it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            


Script finalizado. Rangos: Cielo (<500), Cultivo (1000-1300), Suelo (>1900).
